# Space Debris Collision Risk Predictor — Model Training
Run cells **top to bottom**. GPU kicks in at Cell 7.

---
## Cell 1 — Check Environment

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU     : {g.name}')
    print(f'VRAM    : {g.total_memory/1e9:.1f} GB')
    print(f'CUDA ver: {torch.version.cuda}')
else:
    print('WARNING: No GPU — training will be very slow')

---
## Cell 2 — Imports

In [ ]:
import os, sys, time, math, json, random, shutil, logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast          # new API (PyTorch 2.3+)
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
import h5py

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score
)

PROJECT_ROOT = Path('../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from backend.models.transformer import build_model

logging.basicConfig(level=logging.WARNING)   # suppress INFO spam from libraries
print(f'Project root : {PROJECT_ROOT}')
print('All imports OK')

---
## Cell 3 — Config  *(change things here if needed)*

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR   = PROJECT_ROOT / 'data'
TRAIN_H5   = DATA_DIR / 'processed' / 'train.h5'
VAL_H5     = DATA_DIR / 'processed' / 'val.h5'
TEST_H5    = DATA_DIR / 'processed' / 'test.h5'
MODELS_DIR = DATA_DIR / 'models'
TB_DIR     = MODELS_DIR / 'runs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
TB_DIR.mkdir(parents=True, exist_ok=True)

# ── Model (matches PRD spec) ───────────────────────────────────────────────
MODEL_CFG = dict(
    input_features=22, d_model=256, n_heads=8,
    n_encoder_layers=6, d_feedforward=1024,
    dropout=0.1, max_seq_len=256, n_classes=3
)

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE          = 256
NUM_EPOCHS          = 50
EARLY_STOP_PATIENCE = 15
GRAD_CLIP_NORM      = 1.0
NUM_WORKERS         = 0       # MUST be 0 on Windows in Jupyter
MIXED_PRECISION     = True    # FP16 — set False only if you get NaN losses
SEED                = 42

# ── Optimizer ─────────────────────────────────────────────────────────────
LR_INIT      = 1e-4
WEIGHT_DECAY = 0.01
MAX_LR       = 1e-3
PCT_START    = 0.3

# ── Misc ──────────────────────────────────────────────────────────────────
KEEP_BEST_N  = 3
LOG_EVERY_N  = 50
N_CLASSES    = 3
KL_WEIGHT    = 0.1

print('Config OK')
for p, label in [(TRAIN_H5,'train'), (VAL_H5,'val'), (TEST_H5,'test')]:
    mb = p.stat().st_size / 1e6 if p.exists() else 0
    print(f'  {label:6s}: {p.name}  exists={p.exists()}  ({mb:.0f} MB compressed)')

---
## Cell 4 — Seed

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f'Seed = {SEED}')

---
## Cell 5 — Load Data
Loads HDF5 files into RAM as **float16** (~476 MB total) then creates DataLoaders.  
Expected time: **30–60 seconds**. GPU is NOT used here — that starts at Cell 7.

In [ ]:
class ConjunctionDataset(Dataset):
    def __init__(self, h5_path, augment=False):
        self.augment = augment
        print(f'  Loading {Path(h5_path).name} ...', end=' ', flush=True)
        t0 = time.time()
        with h5py.File(str(h5_path), 'r') as f:
            self.X  = f['X_traj'][:]    # float16  [N, T, F]
            self.y  = f['y_label'][:]   # int64    [N]
            self.p  = f['y_prob'][:]    # float32  [N]
        self.N = len(self.y)
        print(f'{self.N:,} samples  {self.X.nbytes/1e6:.0f} MB  ({time.time()-t0:.1f}s)')
        uc, cc = np.unique(self.y, return_counts=True)
        names = ['LOW','MEDIUM','HIGH']
        for c, n in zip(uc, cc):
            print(f'    {names[int(c)]}: {n:,} ({n/self.N*100:.1f}%)')

    def __len__(self): return self.N

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx].astype('float32'))  # [T, F]
        if self.augment:
            T = x.shape[0]
            x = x[random.randint(0, max(0, T-120)):]
            x = x + torch.randn_like(x) * 0.01
        return {'x': x, 'label': int(self.y[idx]), 'prob': float(self.p[idx])}


def collate_fn(batch):
    max_T = max(b['x'].shape[0] for b in batch)
    B, F  = len(batch), batch[0]['x'].shape[1]
    x_pad = torch.zeros(B, max_T, F)
    mask  = torch.ones(B, max_T+1, dtype=torch.bool)   # True=ignore
    for i, b in enumerate(batch):
        T = b['x'].shape[0]
        x_pad[i, :T] = b['x']
        mask[i, 1:T+1] = False
    mask[:, 0] = False   # CLS always visible
    return {
        'x'    : x_pad,
        'label': torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'prob' : torch.tensor([b['prob']  for b in batch], dtype=torch.float32),
        'mask' : mask,
    }


print('Loading HDF5 files into RAM...')
train_ds = ConjunctionDataset(TRAIN_H5, augment=True)
val_ds   = ConjunctionDataset(VAL_H5)
test_ds  = ConjunctionDataset(TEST_H5)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin    = (device.type == 'cuda')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,
                          num_workers=0, pin_memory=pin, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=0, pin_memory=pin, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=0, pin_memory=pin, collate_fn=collate_fn)

print(f'\nDataLoaders ready  |  device = {device}')
print(f'  train: {len(train_loader):,} batches')
print(f'  val  : {len(val_loader):,} batches')
print(f'  test : {len(test_loader):,} batches')
print('\nCell 5 done. GPU starts at Cell 7.')

---
## Cell 6 — EDL Loss Function

In [ ]:
def edl_mse_loss(alpha, y, epoch, n_classes=3, kl_weight=0.1):
    y_oh = F.one_hot(y, n_classes).float()
    S    = alpha.sum(-1, keepdim=True)
    p    = alpha / S
    mse  = ((y_oh - p)**2).sum(-1).mean()
    var  = (alpha*(S-alpha)) / (S**2*(S+1))
    var  = var.sum(-1).mean()
    at   = y_oh + (1-y_oh)*alpha
    sa   = at.sum(-1)
    kl   = (
        torch.lgamma(sa)
        - torch.lgamma(torch.tensor(float(n_classes), device=alpha.device))
        - torch.lgamma(at).sum(-1)
        + ((at-1)*(torch.digamma(at)-torch.digamma(sa.unsqueeze(-1)))).sum(-1)
    ).mean()
    w     = min(1.0, epoch/10.0) * kl_weight
    total = mse + var + w*kl
    return total, {'total': total.item(), 'mse': mse.item(), 'kl': kl.item()}

# smoke-test
a = torch.ones(4,3)*2
l = torch.tensor([0,1,2,0])
loss, d = edl_mse_loss(a, l, epoch=5)
print(f'EDL loss smoke-test: total={d["total"]:.4f}  mse={d["mse"]:.4f}  kl={d["kl"]:.4f}')
print('Cell 6 done.')

---
## Cell 7 — Build Model  *(GPU activates here)*

In [ ]:
print(f'Device: {device}')
model = build_model({'model': MODEL_CFG}).to(device)
n = model.count_parameters()
print(f'Parameters: {n:,}  ({n/1e6:.1f} M)')

model.eval()
with torch.no_grad():
    x_t = torch.randn(2, 168, 22).to(device)
    ev, al, un, pr = model(x_t)
print(f'Forward pass: evidence={ev.shape}  prob={pr.shape}  prob_sum={pr.sum(-1).tolist()}')

if device.type == 'cuda':
    print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    torch.cuda.empty_cache()
print('Cell 7 done. Model on GPU.')

---
## Cell 8 — Optimizer & Scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR_INIT,
    weight_decay=WEIGHT_DECAY, betas=(0.9,0.999), eps=1e-8
)

total_steps = len(train_loader) * NUM_EPOCHS
scheduler   = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, total_steps=total_steps,
    pct_start=PCT_START, anneal_strategy='cos',
    div_factor=10.0, final_div_factor=1e6
)

USE_AMP = MIXED_PRECISION and device.type == 'cuda'
scaler  = GradScaler('cuda', enabled=USE_AMP)

print(f'Optimizer    : AdamW  lr={LR_INIT}  wd={WEIGHT_DECAY}')
print(f'Scheduler    : OneCycleLR  max_lr={MAX_LR}  steps={total_steps:,}')
print(f'Mixed prec.  : {USE_AMP}')
print('Cell 8 done.')

---
## Cell 9 — Checkpoint Manager

In [ ]:
class CheckpointManager:
    def __init__(self, save_dir, keep_n=3):
        self.d = Path(save_dir); self.d.mkdir(parents=True, exist_ok=True)
        self.n = keep_n; self.ckpts = []   # [(auc, path)]

    def save(self, state, epoch, auc):
        p = self.d / f'ckpt_ep{epoch:03d}_auc{auc:.4f}.pth'
        torch.save(state, p)
        self.ckpts.append((auc, p))
        self.ckpts.sort(key=lambda t: t[0], reverse=True)
        while len(self.ckpts) > self.n:
            _, old = self.ckpts.pop()
            old.exists() and old.unlink()
        if auc == self.ckpts[0][0]:
            shutil.copyfile(p, self.d/'best_model.pth')
            print(f'  >> New best  AUC={auc:.4f}  -> best_model.pth')
        torch.save(state, self.d/'last.pth')

    @property
    def best(self): return self.ckpts[0][0] if self.ckpts else 0.0

ckpt_mgr = CheckpointManager(MODELS_DIR, keep_n=KEEP_BEST_N)
print(f'Checkpoint manager -> {MODELS_DIR}')

---
## Cell 10 — Evaluate Function + Smoke Test

In [ ]:
@torch.no_grad()
def evaluate(model, loader, epoch):
    model.eval()
    labels_all, preds_all, probs_all = [], [], []
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        x  = batch['x'].to(device)
        lb = batch['label'].to(device)
        mk = batch['mask'].to(device)
        with autocast('cuda', enabled=USE_AMP):
            ev, al, un, pr = model(x, src_key_padding_mask=mk)
            loss, _        = edl_mse_loss(al, lb, epoch, N_CLASSES, KL_WEIGHT)
        total_loss += loss.item(); n_batches += 1
        preds_all.extend(pr.argmax(-1).cpu().numpy())
        labels_all.extend(lb.cpu().numpy())
        probs_all.append(pr.cpu().numpy())

    yt = np.array(labels_all)
    yp = np.array(preds_all)
    ys = np.vstack(probs_all)
    acc = accuracy_score(yt, yp)
    pr_, rc, f1, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)
    try:    auc = roc_auc_score(yt, ys, multi_class='ovr', average='macro')
    except: auc = 0.0
    return {'loss': total_loss/max(n_batches,1), 'accuracy': acc,
            'precision': pr_, 'recall': rc, 'f1': f1, 'auc': auc}

print('Running one eval pass on val set (GPU smoke-test)...')
m = evaluate(model, val_loader, 0)
print(f'  loss={m["loss"]:.4f}  acc={m["accuracy"]:.4f}  auc={m["auc"]:.4f}')
if device.type == 'cuda':
    print(f'  GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Cell 10 done. Ready to train.')

---
## Cell 11 — Training Loop
> RTX 5080: expect ~15–20 min/epoch. Watch the `tqdm` bar for per-batch progress.

In [ ]:
writer        = SummaryWriter(str(TB_DIR))
best_auc      = 0.0
patience_cnt  = 0
global_step   = 0
history       = []

print(f'Training start  |  {NUM_EPOCHS} epochs  |  early-stop patience={EARLY_STOP_PATIENCE}')
print(f'Device: {device}  |  Mixed precision: {USE_AMP}')
print('='*70)

for epoch in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    model.train()
    loss_sum, n_seen = 0.0, 0

    pbar = tqdm(train_loader, desc=f'Ep {epoch:02d}/{NUM_EPOCHS}',
                leave=False, unit='batch', dynamic_ncols=True)

    for batch in pbar:
        x  = batch['x'].to(device, non_blocking=True)
        lb = batch['label'].to(device, non_blocking=True)
        mk = batch['mask'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            ev, al, un, pr = model(x, src_key_padding_mask=mk)
            loss, ld       = edl_mse_loss(al, lb, epoch, N_CLASSES, KL_WEIGHT)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        loss_sum += loss.item() * len(lb)
        n_seen   += len(lb)
        global_step += 1

        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.1e}')

        if global_step % LOG_EVERY_N == 0:
            writer.add_scalar('train/loss',    loss.item(),      global_step)
            writer.add_scalar('train/lr',      scheduler.get_last_lr()[0], global_step)
            writer.add_scalar('train/mse',     ld['mse'],        global_step)
            writer.add_scalar('train/kl',      ld['kl'],         global_step)

    pbar.close()
    train_loss = loss_sum / max(n_seen, 1)

    vm = evaluate(model, val_loader, epoch)
    elapsed = time.time() - t0

    print(
        f'Ep {epoch:3d}/{NUM_EPOCHS} | '
        f'trn={train_loss:.4f} | '
        f'val_loss={vm["loss"]:.4f} | '
        f'acc={vm["accuracy"]:.4f} | '
        f'auc={vm["auc"]:.4f} | '
        f'f1={vm["f1"]:.4f} | '
        f'lr={scheduler.get_last_lr()[0]:.1e} | '
        f'{elapsed:.0f}s'
    )

    for k,v in vm.items(): writer.add_scalar(f'val/{k}', v, epoch)
    writer.add_scalar('train/loss_epoch', train_loss, epoch)

    state = {
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'val_metrics': vm, 'train_loss': train_loss, 'config': {'model': MODEL_CFG},
    }
    ckpt_mgr.save(state, epoch, vm['auc'])

    if vm['auc'] > best_auc:
        best_auc, patience_cnt = vm['auc'], 0
    else:
        patience_cnt += 1
        print(f'  patience {patience_cnt}/{EARLY_STOP_PATIENCE}')
        if patience_cnt >= EARLY_STOP_PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    history.append({'epoch': epoch, 'train_loss': train_loss, **vm})

writer.close()
print('='*70)
print(f'Done!  Best val AUC: {best_auc:.4f}')
print(f'Best model: {MODELS_DIR}/best_model.pth')

---
## Cell 12 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

if not history:
    print('No history yet — run Cell 11 first.')
else:
    ep  = [h['epoch']      for h in history]
    tl  = [h['train_loss'] for h in history]
    vl  = [h['loss']       for h in history]
    acc = [h['accuracy']   for h in history]
    auc = [h['auc']        for h in history]
    f1  = [h['f1']         for h in history]

    fig, ax = plt.subplots(1,3,figsize=(15,4))
    ax[0].plot(ep,tl,label='train'); ax[0].plot(ep,vl,label='val')
    ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep,acc,color='seagreen'); ax[1].axhline(.94,ls='--',c='red',label='target 94%')
    ax[1].set_title('Val Accuracy'); ax[1].legend(); ax[1].set_ylim(0,1.05); ax[1].grid(alpha=.3)
    ax[2].plot(ep,auc,label='AUC'); ax[2].plot(ep,f1,ls='--',label='F1')
    ax[2].axhline(.98,ls='--',c='red',label='target 0.98')
    ax[2].set_title('AUC-ROC & F1'); ax[2].legend(); ax[2].set_ylim(0,1.05); ax[2].grid(alpha=.3)
    plt.suptitle('Training History', fontweight='bold')
    plt.tight_layout()
    out = MODELS_DIR/'training_curves.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved -> {out}')

---
## Cell 13 — Test Set Evaluation

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
from sklearn.metrics import auc as sk_auc

BEST = MODELS_DIR / 'best_model.pth'
if not BEST.exists():
    print('best_model.pth not found — run Cell 11 first.'); raise SystemExit

ck = torch.load(BEST, map_location=device, weights_only=True)
model.load_state_dict(ck['model_state_dict'])
print(f'Loaded best model (epoch {ck["epoch"]}, val AUC={ck["val_metrics"]["auc"]:.4f})')

model.eval()
labels_all, preds_all, probs_all, unc_all = [], [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference'):
        x  = batch['x'].to(device)
        mk = batch['mask'].to(device)
        with autocast('cuda', enabled=USE_AMP):
            ev, al, un, pr = model(x, src_key_padding_mask=mk)
        preds_all.extend(pr.argmax(-1).cpu().numpy())
        labels_all.extend(batch['label'].numpy())
        probs_all.append(pr.cpu().numpy())
        unc_all.extend(un.cpu().numpy())

yt = np.array(labels_all); yp = np.array(preds_all); ys = np.vstack(probs_all)
acc = accuracy_score(yt, yp)
pr_, rc, f1, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)
try:    auc_val = roc_auc_score(yt, ys, multi_class='ovr', average='macro')
except: auc_val = 0.0

print('\n' + '='*45 + '\nTEST SET RESULTS\n' + '='*45)
print(f'  Accuracy  : {acc:.4f}   (target >= 0.94)')
print(f'  AUC-ROC   : {auc_val:.4f}   (target >= 0.98)')
print(f'  Precision : {pr_:.4f}')
print(f'  Recall    : {rc:.4f}')
print(f'  F1 macro  : {f1:.4f}')
print(f'  Avg uncert: {np.mean(unc_all):.4f}')
print()
print(classification_report(yt, yp, target_names=['LOW','MEDIUM','HIGH']))

# Confusion matrix
fig, axes = plt.subplots(1,2,figsize=(13,5))
cm = confusion_matrix(yt, yp)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['LOW','MEDIUM','HIGH'], yticklabels=['LOW','MEDIUM','HIGH'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (acc={acc:.3f})')

# ROC curves
for i, (name, col) in enumerate(zip(['LOW','MEDIUM','HIGH'], ['steelblue','seagreen','tomato'])):
    fpr,tpr,_ = roc_curve((yt==i).astype(int), ys[:,i])
    axes[1].plot(fpr, tpr, color=col, lw=2, label=f'{name} (AUC={sk_auc(fpr,tpr):.3f})')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curves (One-vs-Rest)'); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
out = MODELS_DIR/'test_results.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved -> {out}')

with open(MODELS_DIR/'test_metrics.json','w') as fp:
    json.dump({'accuracy':float(acc),'auc_roc':float(auc_val),'f1':float(f1),
               'precision':float(pr_),'recall':float(rc),'n_test':len(yt)}, fp, indent=2)
print(f'Metrics saved -> {MODELS_DIR}/test_metrics.json')

---
## Cell 14 — Resume Training *(only if interrupted)*
Run this cell instead of Cell 8, then re-run Cell 11.

In [ ]:
LAST = MODELS_DIR / 'last.pth'
if not LAST.exists():
    print('No last.pth found — nothing to resume.')
else:
    ck = torch.load(LAST, map_location=device, weights_only=True)
    model.load_state_dict(ck['model_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    scheduler.load_state_dict(ck['scheduler_state_dict'])
    scaler.load_state_dict(ck['scaler_state_dict'])
    best_auc = ck['val_metrics']['auc']
    print(f'Resumed from epoch {ck["epoch"]}  (best AUC so far: {best_auc:.4f})')
    print('Now re-run Cell 11.')